# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Utsabsinha19/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

I will review two findings from the FlyRank research paper and examine whether the label definition and validation design support the strength of each claim. For each finding, I will identify whether the label is based on an observed outcome or a defined proxy, and whether the validation design appropriately separates training information from evaluation information.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Paper findings review:")
print("Finding 1: label source and validation design need to be checked.")
print("Finding 2: label source and validation design need to be checked.")
print("No stronger claim will be made without verifying the paper methodology.")

Paper findings review:
Finding 1: label source and validation design need to be checked.
Finding 2: label source and validation design need to be checked.
No stronger claim will be made without verifying the paper methodology.


## 2. My model under an honest split (before/after)

I will evaluate the model using a client-grouped split so that pages from the same client do not appear in both training and test data. This gives a more honest estimate of how the model behaves on unseen clients. I will compare the result with the earlier model result using the same Precision@50 metric.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# Load dataset
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Dataset:", df.shape)

# Recent change
df["impression_change_pct"] = np.where(
    df["impressions_prev_30d"] > 0,
    (
        df["impressions_last_30d"]
        - df["impressions_prev_30d"]
    ) / df["impressions_prev_30d"],
    0
)

# Same provisional proxy used in ML-08
df["review_proxy_score"] = (
    0.40 * df["impressions_90d"].rank(pct=True)
    + 0.35 * df["avg_position"].rank(pct=True)
    + 0.25 * (-df["impression_change_pct"]).rank(pct=True)
)

threshold = df["review_proxy_score"].quantile(0.75)

df["review_label"] = (
    df["review_proxy_score"] >= threshold
).astype(int)

# Model features
feature_columns = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update"
]

X = df[feature_columns].copy()
X = X.fillna(X.median(numeric_only=True))

y = df["review_label"]
groups = df["client_id"]

# Client-grouped split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Train model
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

scores = model.predict_proba(X_test)[:, 1]

# Precision@50
order = np.argsort(scores)[::-1][:50]

precision_at_50 = y_test.iloc[order].mean()

print("Training rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Training clients:", df.iloc[train_idx]["client_id"].nunique())
print("Test clients:", df.iloc[test_idx]["client_id"].nunique())

print(
    "Client overlap:",
    len(
        set(df.iloc[train_idx]["client_id"])
        &
        set(df.iloc[test_idx]["client_id"])
    )
)

print("Honest-split Precision@50:",
      round(precision_at_50, 4))

Dataset: (30000, 44)
Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0
Honest-split Precision@50: 0.72


The validation result measures performance against a provisional proxy label constructed from observed search signals. Because some related signals are available to the model, this should not be interpreted as proof of real-world content-improvement accuracy. A stronger future evaluation would use an independently observed outcome or human-reviewed label.

## 3. Leakage audit

I audited the final feature set for identifiers, target-derived fields, and future-looking information. Client and content identifiers are excluded from the feature matrix. Trend fields and direct outcome-derived fields used to construct the review proxy are also excluded. The 30-day and 90-day fields represent observed historical windows, but their aggregation and possible overlap limit how strongly the results can be interpreted.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== FINAL FEATURE LEAKAGE AUDIT ===")

excluded_identifiers = [
    "content_id",
    "client_id"
]

excluded_outcome_fields = [
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_direction",
    "trend_pct"
]

print("Identifier fields in model:")
print([
    col for col in excluded_identifiers
    if col in feature_columns
])

print("\nOutcome/trend fields in model:")
print([
    col for col in excluded_outcome_fields
    if col in feature_columns
])

print("\nFinal feature count:", len(feature_columns))

print("\nPotential future-window fields:")
future_keywords = [
    "future",
    "next",
    "after"
]

print([
    col for col in feature_columns
    if any(k in col.lower() for k in future_keywords)
])

=== FINAL FEATURE LEAKAGE AUDIT ===
Identifier fields in model:
[]

Outcome/trend fields in model:
[]

Final feature count: 24

Potential future-window fields:
[]


## 4. Claim rewrite

Original bold claim: “The Random Forest can identify the pages that need content improvement.”

Safer claim: “In the observed dataset, the Random Forest produced a ranked list of pages using available content and search-performance signals. Under the client-grouped validation split, its Precision@50 was measured at the reported value. This result is directional decision-support and does not prove that the recommended pages require content changes or that the model will generalize to every future dataset.”

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Claim audit:")
print("Observed: model rankings were evaluated on held-out clients.")
print("Measured: Precision@50 was calculated.")
print("Directional: rankings can support review prioritization.")
print("Not claimed: causal impact or guaranteed future performance.")

Claim audit:
Observed: model rankings were evaluated on held-out clients.
Measured: Precision@50 was calculated.
Directional: rankings can support review prioritization.
Not claimed: causal impact or guaranteed future performance.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.